In [1]:
# =============================================================================
# Autor:   Thiago Souza (@engthiago1979-blip)
# Criado:  19-04-2026  |  Revisado: 06-06-2026 (v2)
# Uso:     Consolida dados de ETEs, padroniza nomes, imputa <LQ e avalia VMP
#          (CONAMA 430/11) exportando XLSX.
# v2:      - Caminhos via pathlib (uma única linha p/ trocar de máquina).
#          - Correção da fragmentação de DataFrame (concat único das colunas).
#          - Supressão de warnings DIRECIONADA (não mais global).
# =============================================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.errors import PerformanceWarning

# Suprime APENAS o aviso de fragmentação (mantém os demais visíveis)
warnings.filterwarnings("ignore", category=PerformanceWarning)

# -----------------------------------------------------------------------------
# 0. RAIZ DO PROJETO -> ÚNICA linha a alterar ao mudar de máquina/ambiente
# -----------------------------------------------------------------------------
RAIZ = Path(r"C:\Users\thiago\Documents\estudos_python\etes_nesa")

DIR_BASE = RAIZ / "base_dados"
FILE_NAME = "BD_25RC_3.1 PCAI_EFLUENTE SANITARIO (31122026)-2.xlsx"
FILE_PATH = DIR_BASE / FILE_NAME
OUTPUT_PATH = DIR_BASE / "ETEs_NorteEnergia_Processado_v1.xlsx"

# -----------------------------------------------------------------------------
# 1. CONFIGURAÇÕES INSTITUCIONAIS E REGULATÓRIAS
# -----------------------------------------------------------------------------
NESA_COLORS = {
    'primary': '#009CA7', 'secondary_1': '#004851', 'secondary_2': '#006D79',
    'highlight': '#B9F8FF', 'vmp': '#F25A3C', 'ok': '#52BD8B',
}

# VMPs - CONAMA 430/11 (Art. 16)
VMP_CONAMA_430 = {
    'PH': (5.0, 9.0),
    'TEMPERATURA_DO_EFLUENTE': 40.0,
    'MATERIAIS_SEDIMENTAVEIS': 1.0,
    'DEMANDA_BIOQUIMICA_DE_OXIGENIO': 120.0,
    'OLEOS_E_GRAXAS': 50.0,
}

# -----------------------------------------------------------------------------
# 2. FUNÇÕES DE TRANSFORMAÇÃO E LIMPEZA
# -----------------------------------------------------------------------------
def filtrar_etes_alvo(df):
    """Filtra as ETEs alvo com base na coluna de descrição (índice 2)."""
    nome_coluna_ete = df.columns[2]
    etes_alvo = ['ETE COMPACTA', 'ETE PM', 'ETE 1', 'ETE 2', 'ETE 01', 'ETE 02']
    mask = df[nome_coluna_ete].astype(str).str.upper().str.strip().isin(etes_alvo)
    return df[mask].copy()

def normalizar_nomes_colunas(df):
    """Higieniza e padroniza nomes das colunas para permitir o merge perfeito."""
    novas_colunas = {}
    for i, col in enumerate(df.columns):
        if i < 4:
            novas_colunas[col] = col  # Mantém metadados intactos
            continue
        nome_limpo = str(col).upper()
        nome_limpo = re.sub(r'\(?MG/L\)?|\(?US/CM\)?|\(?NMP/ML\)?|\(?NMP/100ML\)?|°C|µG/L', '', nome_limpo)
        nome_limpo = re.sub(r'\(IN LOCO\)', '', nome_limpo)
        nome_limpo = re.sub(r'[^\w\s]', '', nome_limpo)
        nome_limpo = re.sub(r'\s+', '_', nome_limpo.strip())
        if 'XILENO' in nome_limpo: nome_limpo = 'XILENO_TOTAL'
        if 'DBO' in nome_limpo: nome_limpo = 'DEMANDA_BIOQUIMICA_DE_OXIGENIO'
        if 'DQO' in nome_limpo: nome_limpo = 'DEMANDA_QUIMICA_DE_OXIGENIO'
        novas_colunas[col] = nome_limpo
    return df.rename(columns=novas_colunas)

def separar_metadados_parametros(df):
    """Retorna listas de colunas: metadados (0-3) e parâmetros (4+)."""
    return df.columns[:4].tolist(), df.columns[4:].tolist()

def tratar_dados_censurados_vetorizado(df, colunas_parametros):
    """Substitui valores <LQ por LQ/sqrt(2) e cria flags de rastreabilidade.

    v2: as flags _IMPUTADO_LQ são acumuladas e adicionadas em UM único
    pd.concat ao final, eliminando o PerformanceWarning de fragmentação.
    """
    df_clean = df.copy()
    flags = {}
    for col in colunas_parametros:
        if df_clean[col].dtype == object:
            mask_censurado = df_clean[col].astype(str).str.contains('<', na=False)
            serie_limpa = df_clean[col].astype(str).str.replace(',', '.', regex=False)
            serie_numerica = serie_limpa.str.extract(r'(\d+\.?\d*)')[0].astype(float)
            df_clean[col] = np.where(
                mask_censurado,
                serie_numerica / np.sqrt(2),
                pd.to_numeric(serie_limpa, errors='coerce'),
            )
            flags[col + '_IMPUTADO_LQ'] = mask_censurado
        else:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    if flags:
        df_clean = pd.concat([df_clean, pd.DataFrame(flags, index=df_clean.index)], axis=1)
    return df_clean.copy()  # desfragmenta o frame final

def status_conformidade_vmp(serie_valores, vmp):
    """Retorna o array de status (Conforme/Não Conforme/Sem Dado) p/ um parâmetro."""
    if isinstance(vmp, tuple):
        condicao_ok = (serie_valores >= vmp[0]) & (serie_valores <= vmp[1])
    else:
        condicao_ok = serie_valores <= vmp
    return np.where(
        serie_valores.isna(), 'Sem Dado',
        np.where(condicao_ok, 'Conforme', 'Não Conforme'),
    )

# -----------------------------------------------------------------------------
# 3. PIPELINE DE EXECUÇÃO
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    print(f"[*] Lendo banco de dados em: {FILE_PATH}")
    try:
        df_bm_raw = pd.read_excel(FILE_PATH, sheet_name="BM - SAIDA ETE")
        df_pm_raw = pd.read_excel(FILE_PATH, sheet_name="PM - SAIDA ETE")

        # 1+2. Filtro e Normalização
        df_bm = normalizar_nomes_colunas(filtrar_etes_alvo(df_bm_raw))
        df_pm = normalizar_nomes_colunas(filtrar_etes_alvo(df_pm_raw))

        # 3. Concatenação (Unificando BM e PM)
        df_consolidado = pd.concat([df_bm, df_pm], ignore_index=True)

        # 4. Separação e Tratamento
        cols_meta, cols_param = separar_metadados_parametros(df_consolidado)
        df_tratado = tratar_dados_censurados_vetorizado(df_consolidado, cols_param)

        # 5. Aplicação de VMP -> acumula colunas de status e concatena de uma vez
        status_cols = {
            param + "_STATUS_VMP": status_conformidade_vmp(df_tratado[param], vmp)
            for param, vmp in VMP_CONAMA_430.items()
            if param in df_tratado.columns
        }
        if status_cols:
            df_tratado = pd.concat(
                [df_tratado, pd.DataFrame(status_cols, index=df_tratado.index)], axis=1
            )

        # 6. Exportação padronizada (XLSX)
        OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
            df_tratado.to_excel(writer, sheet_name="Efluentes_Consolidado", index=False)

        print("[+] Processamento concluído com sucesso!")
        print(f"[+] Arquivo gerado: {OUTPUT_PATH}")
        print("[!] Os dados não conformes estão mapeados nas colunas '_STATUS_VMP'.")

    except FileNotFoundError:
        print(f"[!] ERRO: Arquivo não encontrado: {FILE_PATH}. Verifique o nome exato da planilha.")
    except Exception as e:
        print(f"[!] ERRO INESPERADO: {e}")


[*] Lendo banco de dados em: C:\Users\thiago\Documents\estudos_python\etes_nesa\base_dados\BD_25RC_3.1 PCAI_EFLUENTE SANITARIO (31122026)-2.xlsx
[+] Processamento concluído com sucesso!
[+] Arquivo gerado: C:\Users\thiago\Documents\estudos_python\etes_nesa\base_dados\ETEs_NorteEnergia_Processado_v1.xlsx
[!] Os dados não conformes estão mapeados nas colunas '_STATUS_VMP'.
